### Statewide zone mapping

Foundation notebook for the statewide extension. Rafael uploaded `tract_to_bus_mapping_statewide_ExportTable.csv` (6,896 tracts, 3,426 buses, all 254 Texas counties) — a genuinely new, denser synthetic grid, not an extension of the original 200-bus Coast case (a 1km nearest-substation match against the old Coast file only recovers 72/200 buses, confirming this is a different grid, not a superset).

This notebook:
1. Builds an authoritative county -> IECC climate zone crosswalk for Texas, using NREL's own `spatial_tract_lookup_table.csv` (not a third-party map — avoids repeating the exact kind of geography mismatch that caused the original weather-station bug).
2. Joins it to the new statewide bus mapping to get bus counts and county lists per zone.
3. Computes per-zone, population-weighted county weather weights (same method as `weather_correction.ipynb`, generalized).
4. Saves everything the five `residential_{ZONE}.ipynb` notebooks need, so none of them re-derive it.

In [1]:
# env1 3.11.13 (repo default)
import pandas as pd
import numpy as np
from pathlib import Path
import requests

exports = Path("../../loadcurves/exports")
out_dir = Path("..")
assert exports.exists(), f"Expected {exports.resolve()}"


#### 1. County -> IECC climate zone (authoritative, from NREL)

`spatial_tract_lookup_table.csv` ships with the ResStock dataset itself (`geographic_information/`), so this crosswalk is guaranteed consistent with the same NREL archetype data the pipeline downloads — not a third-party map that could silently disagree.

In [2]:
lookup_url = (
    "https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/"
    "end-use-load-profiles-for-us-building-stock/2022/resstock_amy2018_release_1/"
    "geographic_information/spatial_tract_lookup_table.csv"
)
lookup_path = out_dir / "spatial_tract_lookup_table.csv"
if not lookup_path.exists():
    r = requests.get(lookup_url, timeout=120)
    r.raise_for_status()
    lookup_path.write_bytes(r.content)

lookup = pd.read_csv(lookup_path, dtype=str)
tx = lookup[lookup["state_abbreviation"] == "TX"].copy()
tx["county_fips"] = tx["nhgis_2010_county_gisjoin"].str[4:7]

county_zone = tx.groupby("county_fips")["iecc_2012_climate_zone"].agg(lambda s: s.value_counts().to_dict())
n_split = (county_zone.apply(len) > 1).sum()
print(f"TX tracts in NREL lookup: {len(tx)}")
print(f"Counties split across multiple IECC zones: {n_split} / {tx['county_fips'].nunique()} (expect 0)")

county_to_zone = tx.groupby("county_fips")["iecc_2012_climate_zone"].agg(lambda s: s.value_counts().idxmax())
county_to_zone.name = "iecc_zone"
print("\nCounties per zone:")
print(county_to_zone.value_counts())

county_to_zone.to_csv(out_dir / "county_to_iecc_zone.csv", header=True)


TX tracts in NREL lookup: 5265
Counties split across multiple IECC zones: 0 / 254 (expect 0)

Counties per zone:
iecc_zone
2A    79
3B    70
3A    63
4B    28
2B    14
Name: count, dtype: int64


#### 2. Bus -> zone, and tract -> bus **area weights** (new ArcGIS exports)

Two files replace the old `tract_to_bus_mapping_statewide_ExportTable.csv`, which matched each
tract to a single nearest bus and left most of the grid with no load at all.

- **`texas7k_bus_with_climate_zone.csv`** — the authoritative bus table: all **6,717** Texas 7k
  buses with substation lat/lon, nominal kV, the case's own `Load_MW`/`Gen_MW`, the ERCOT
  `Area_Name`, county FIPS and IECC zone. The zone no longer has to be inferred by majority vote
  over whichever tracts happened to pick that bus; and buses that never won a tract (generation-
  only or lightly loaded ones) are now present instead of silently missing. Its `iecc_zone` column
  agrees with the NREL-derived county crosswalk of step 1 on **100% of the 6,717 buses**, so the
  two independent geographies confirm each other. `Load_MW` gives a per-bus reference against which
  the reconstruction can be calibrated, and `Area_Name` gives the ERCOT weather zone needed to
  validate against published ERCOT zonal load.
- **`tract_to_bus_mapping_statewide1.csv`** — each 2020 census tract **subdivided into 50
  equal-area parts** (verified: the 50 `Shape_Area` values per tract are equal to 1e-6 and sum to
  `ALAND`), each part spatially joined to its nearest bus. So
  `weight(tract, bus) = parts won / 50`, in 2% increments, and a tract's households are spread
  across every bus that serves part of it. This is exactly the spatial-assignment refinement the
  paper lists as future work (§IV-F): it replaces the winner-takes-all nearest-bus rule.

In [3]:
N_PARTS = 50
repo_root = out_dir.resolve().parent  # repo root

# --- 2a. authoritative bus table -------------------------------------------------
bus = pd.read_csv(repo_root / "texas7k_bus_with_climate_zone.csv", dtype={"COUNTYFP": str})
bus["COUNTYFP"] = bus["COUNTYFP"].str.zfill(3)
bus["Load_MW"] = bus["Load_MW"].fillna(0.0)
bus["Gen_MW"] = bus["Gen_MW"].fillna(0.0)
bus = bus.rename(columns={"Substation_Latitude": "lat", "Substation_Longitude": "lon"})
bus = bus[["Bus_Number", "lat", "lon", "Name_1", "Nom_kV", "Load_MW", "Gen_MW",
           "Area_Name", "COUNTYFP", "iecc_zone"]].drop_duplicates("Bus_Number")

# cross-check: does the shapefile's own zone agree with the NREL county crosswalk of step 1?
agree = (bus["COUNTYFP"].map(county_to_zone.to_dict()) == bus["iecc_zone"]).mean()
print(f"Bus zone vs NREL county crosswalk agreement: {agree:.2%} of {len(bus)} buses")

bus.to_csv(out_dir / "bus_zone_master.csv", index=False)
print(bus.groupby("iecc_zone").agg(buses=("Bus_Number", "size"),
                                   load_buses=("Load_MW", lambda s: (s > 0).sum()),
                                   case_MW=("Load_MW", "sum")))

# --- 2b. tract -> bus area weights from the 50-part subdivision -------------------
raw = pd.read_csv(repo_root / "tract_to_bus_mapping_statewide1.csv")
raw.columns = [c.lstrip("\ufeff") for c in raw.columns]
T = "tl_2020_48_tract_Project_"
raw["GEOID"] = raw[T + "GEOID"].astype(str).str.zfill(11)
raw["Bus_Number"] = raw["Number"].astype(float).round().astype(int)

assert (raw.groupby("GEOID").size() == N_PARTS).all(), "expected exactly 50 parts per tract"

w = (raw.groupby(["GEOID", "Bus_Number"]).size() / N_PARTS).rename("weight").reset_index()
w["COUNTYFP"] = w["GEOID"].str[2:5]
w["TRACTCE"] = w["GEOID"].str[5:]
w = w.merge(bus[["Bus_Number", "iecc_zone", "Area_Name"]], on="Bus_Number", how="left")
assert np.allclose(w.groupby("GEOID")["weight"].sum(), 1.0)
w.to_csv(out_dir / "tract_bus_weights.csv", index=False)

split = w.groupby("GEOID").size()
load_buses = set(bus.loc[bus["Load_MW"] > 0, "Bus_Number"])
print(f"\ntract_bus_weights: {w['GEOID'].nunique()} tracts -> {w['Bus_Number'].nunique()} buses, "
      f"{len(w)} pairs (mean {split.mean():.2f} buses/tract, max {split.max()})")
print(f"  tracts split across >1 bus: {(split > 1).sum()} / {len(split)} ({(split > 1).mean():.1%})")
print(f"  case-load buses reached: {len(load_buses & set(w['Bus_Number']))} / {len(load_buses)}"
      f"  (old single-nearest-bus file reached 3,426 buses in total)")
print(f"  tracts straddling an IECC zone boundary: {(w.groupby('GEOID')['iecc_zone'].nunique() > 1).sum()}")


Bus zone vs NREL county crosswalk agreement: 100.00% of 6717 buses
           buses  load_buses   case_MW
iecc_zone                             
2A          3580        2385  40008.24
2B           246         189   1407.51
3A          2142        1610  26686.15
3B           625         293   5343.88
4B           124          64   1220.81



tract_bus_weights: 6896 tracts -> 6359 buses, 24499 pairs (mean 3.55 buses/tract, max 29)
  tracts split across >1 bus: 5790 / 6896 (84.0%)
  case-load buses reached: 4518 / 4541  (old single-nearest-bus file reached 3,426 buses in total)
  tracts straddling an IECC zone boundary: 131


#### 3b. Fix: re-fetch Census B25024 using 2020 tract boundaries

**Bug found while building the first zone notebook (2A)**: 740/1883 buses came out with zero building count. Root cause: the new statewide `tract_to_bus_mapping_statewide_ExportTable.csv` was built from **2020 Census tract boundaries** (Harris County = 1,115 tracts), but the cached `census_b25024_state_48_tracts.csv` (used successfully for the original Coast pipeline) is on **2010 tract boundaries** (Harris County = 786 tracts, exactly matching the old Coast-only `tract_to_bus_mapping.csv` — that file was never wrong, it's simply a different, older vintage). Only 485/1115 Harris tracts overlap between the two vintages — a tract-boundary redistricting mismatch, the same class of bug as the original PUMA-vs-county weather station issue, just in a different join.\n\nFix: fetch a **new** Census B25024 file on 2020 tract boundaries (ACS5 2022 vintage, confirmed to return exactly 1,115 Harris tracts) for the statewide extension only — the original `loadcurves/exports/census_b25024_state_48_tracts.csv` is left untouched since it correctly matches the original Coast-only files.

In [4]:
import os

def get_census_api_key():
    key = os.environ.get("CENSUS_API_KEY", "").strip()
    if key:
        return key
    key_file = Path("../../census_api_key.txt")
    if key_file.exists():
        return key_file.read_text().strip()
    return ""


census_2020_path = out_dir / "census_b25024_state_48_tracts_2020vintage.csv"
if not census_2020_path.exists():
    params = {
        "get": "NAME,group(B25024)",
        "for": "tract:*",
        "in": "state:48",
        "key": get_census_api_key(),
    }
    r = requests.get("https://api.census.gov/data/2022/acs/acs5", params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    census_2020 = pd.DataFrame(data[1:], columns=data[0])
    cols_to_keep = ["NAME", "state", "county", "tract"] + [c for c in census_2020.columns if c.endswith("E") and "B25024" in c]
    census_2020 = census_2020[cols_to_keep]
    census_2020.to_csv(census_2020_path, index=False)

census_2020 = pd.read_csv(census_2020_path, dtype={"county": str, "tract": str})
census_2020["county"] = census_2020["county"].str.zfill(3)
harris_check = (census_2020["county"] == "201").sum()
print(f"2020-vintage census: {len(census_2020)} TX tracts total, {harris_check} in Harris County (expect 1115)")


2020-vintage census: 6896 TX tracts total, 1115 in Harris County (expect 1115)


#### 3. Archetype counts per bus, and per-zone county weather weights

With area weights the allocation is a weighted sum rather than a groupby: a tract's housing units
are multiplied by each bus's share. Because every tract's weights sum to 1, **100% of Texas
housing units are conserved** — nothing is dropped and nothing is double-counted.

The county weather weights are then rebuilt from the same allocation: a county's weight inside a
zone is the share of that zone's housing units its tracts actually contribute (previously it was
the share of *dwellings in counties whose tracts happened to pick a bus in the zone*, a coarser
proxy). Same method validated for Coast in `weather_correction.ipynb`, now consistent with the
spatial split.

In [5]:
nrel_mapping = {
    "single-family_detached": ["B25024_002E"],
    "single-family_attached": ["B25024_003E"],
    "mobile_home": ["B25024_010E"],
    "multi-family_with_2_-_4_units": ["B25024_004E", "B25024_005E"],
    "multi-family_with_5plus_units": ["B25024_006E", "B25024_007E", "B25024_008E", "B25024_009E"],
}
archetypes = list(nrel_mapping)

# 2020-tract-vintage census — see section 3b below for why this vintage and not the cached one.
census_2020 = pd.read_csv(out_dir / "census_b25024_state_48_tracts_2020vintage.csv",
                          dtype={"state": str, "county": str, "tract": str})
census_2020["GEOID"] = (census_2020["state"].str.zfill(2) + census_2020["county"].str.zfill(3)
                        + census_2020["tract"].str.zfill(6))
for cols in nrel_mapping.values():
    for c in cols:
        census_2020[c] = pd.to_numeric(census_2020[c], errors="coerce").fillna(0.0)
for arch, cols in nrel_mapping.items():
    census_2020[arch] = census_2020[cols].sum(axis=1)

alloc = w.merge(census_2020[["GEOID"] + archetypes], on="GEOID", how="left")
alloc[archetypes] = alloc[archetypes].fillna(0.0).mul(alloc["weight"], axis=0)

bus_counts = (alloc.groupby("Bus_Number")[archetypes].sum()
              .reindex(bus["Bus_Number"].values).fillna(0.0).rename_axis("Bus_Number"))
bus_counts = bus_counts.join(bus.set_index("Bus_Number")[["iecc_zone", "COUNTYFP", "Area_Name"]])
bus_counts.to_csv(out_dir / "bus_building_counts_statewide.csv")

census_total = census_2020[archetypes].to_numpy().sum()
allocated = bus_counts[archetypes].to_numpy().sum()
nz = bus_counts[archetypes].sum(axis=1) > 0
print(f"Housing units allocated: {allocated:,.0f} / {census_total:,.0f} census total "
      f"({allocated / census_total:.2%} conserved)")
print(f"Buses with non-zero housing: {nz.sum()} / {len(bus_counts)}; "
      f"case-load buses with zero housing: {((bus.set_index('Bus_Number')['Load_MW'] > 0) & ~nz).sum()}")
print(bus_counts.groupby("iecc_zone")[archetypes].sum().assign(total=lambda d: d.sum(axis=1)).round(0))

# --- per-zone county weather weights, from the same allocation --------------------
alloc["units"] = alloc[archetypes].sum(axis=1)
zone_county = alloc.groupby(["iecc_zone", "COUNTYFP"])["units"].sum()
weights_df = (zone_county / zone_county.groupby(level=0).sum()).rename("weight").reset_index()
weights_df = weights_df.rename(columns={"COUNTYFP": "county_fips"})
weights_df.to_csv(out_dir / "zone_county_weather_weights.csv", index=False)

print("\nCounty weather weights (top 3 per zone):")
for z, g in weights_df.groupby("iecc_zone"):
    top = g.nlargest(3, "weight")
    print(f"  {z}: {len(g)} counties | " + ", ".join(f"{r.county_fips}={r.weight:.1%}" for r in top.itertuples()))


Housing units allocated: 11,628,419 / 11,628,419 census total (100.00% conserved)
Buses with non-zero housing: 6352 / 6717; case-load buses with zero housing: 25
           single-family_detached  single-family_attached  mobile_home  \
iecc_zone                                                                
2A                      4117865.0                181406.0     426595.0   
2B                       132115.0                  4627.0      28221.0   
3A                      2509849.0                106000.0     227157.0   
3B                       624445.0                 26445.0      77070.0   
4B                       156004.0                  4836.0      18075.0   

           multi-family_with_2_-_4_units  multi-family_with_5plus_units  \
iecc_zone                                                                 
2A                              347381.0                      1372906.0   
2B                               11592.0                        17574.0   
3A                 

#### 4. Cross-check candidate: does the new grid's Coast-county subset line up with the already-validated Coast pipeline?

The old 200-bus Coast grid and the new statewide grid do not share bus IDs and are not 1:1 spatially matchable (verified: only 72/200 old buses have a new-grid bus within 1km). A bus-level cross-check is therefore not reliable. Instead, `statewide_summary.ipynb` will do a **county-level aggregate** cross-check: sum the new 2A-zone reconstruction restricted to the original 12 Coast counties, and compare the shape/magnitude to the already-validated Coast-only reconstruction. This cell just saves the list of the 12 original Coast counties for that later comparison.

In [6]:
old_coast_map = pd.read_csv(exports / "tract_to_bus_mapping.csv", dtype={"COUNTYFP": str})
old_coast_map["COUNTYFP"] = old_coast_map["COUNTYFP"].str.zfill(3)
original_coast_counties = sorted(old_coast_map["COUNTYFP"].unique())
print(f"Original 12 Coast counties: {original_coast_counties}")

pd.Series(original_coast_counties, name="county_fips").to_csv(out_dir / "original_coast_counties.csv", index=False)


Original 12 Coast counties: ['039', '057', '071', '157', '167', '201', '239', '321', '339', '469', '473', '481']


#### 5. ZCTA -> bus **area weights** (for the commercial/ComStock extension)

`zcta_to_bus_mapping_statewide_tracts_subdivided100.csv` is the ZCTA counterpart of the tract
file, and it closes the last winner-takes-all join in the pipeline. Each of the **1,933 Texas
ZCTAs is subdivided into 100 equal-area parts** (verified: within a ZCTA the 100 `Shape_Area`
values are equal to ~1e-7 and sum to the ZCTA area), each part joined to its nearest bus, so
`weight(zcta, bus) = parts won / 100` in 1% increments.

Why this matters: the previous file gave each ZCTA entirely to one bus, and reached only **1,598
buses** — barely 15% of the buses in any ERCOT zone carried commercial load, which is what made
the per-zone commercial calibration degenerate (South Central fitted `COM_SCALE = 0`, West 3.62).
The subdivided file reaches **6,329 buses**.

It also has 98 fewer ZCTAs than the old file (1,933 vs 2,031). Those 98 are `706xx` codes —
**Louisiana** ZCTAs that bled across the state line in the old export, and they carry **zero** ZBP
commercial employment. Their loss is a correction, not a gap.

In [7]:
N_ZCTA_PARTS = 100

zraw = pd.read_csv(repo_root / "zcta_to_bus_mapping_statewide_tracts_subdivided100.csv")
zraw.columns = [c.lstrip("\ufeff") for c in zraw.columns]
zraw["ZCTA5CE10"] = zraw["ZCTA5CE10"].astype(str).str.zfill(5)
zraw["Bus_Number"] = zraw["Number"].astype(float).round().astype(int)

assert (zraw.groupby("ZCTA5CE10").size() == N_ZCTA_PARTS).all(), "expected exactly 100 parts per ZCTA"

zw = (zraw.groupby(["ZCTA5CE10", "Bus_Number"]).size() / N_ZCTA_PARTS).rename("weight").reset_index()
zw = zw.merge(bus[["Bus_Number", "iecc_zone", "COUNTYFP", "Area_Name"]], on="Bus_Number", how="left")
assert np.allclose(zw.groupby("ZCTA5CE10")["weight"].sum(), 1.0)
assert zw["iecc_zone"].notna().all()
zw.to_csv(out_dir / "zcta_bus_weights.csv", index=False)

zsplit = zw.groupby("ZCTA5CE10").size()
print(f"zcta_bus_weights: {zw['ZCTA5CE10'].nunique()} ZCTAs -> {zw['Bus_Number'].nunique()} buses, "
      f"{len(zw)} pairs (mean {zsplit.mean():.2f} buses/ZCTA, max {zsplit.max()})")
print(f"  ZCTAs split across >1 bus: {(zsplit > 1).sum()} / {len(zsplit)} ({(zsplit > 1).mean():.1%})")
print(f"  previous nearest-bus file reached 1,598 buses in total")

# majority-weight zone label per ZCTA, kept for anything that needs one label per ZCTA
zcta_zone = (zw.sort_values("weight", ascending=False)
             .drop_duplicates("ZCTA5CE10").set_index("ZCTA5CE10")["iecc_zone"])
zcta_zone.to_frame().reset_index().to_csv(out_dir / "zcta_to_zone.csv", index=False)
print()
print("ZCTAs per zone (majority weight):")
print(zcta_zone.value_counts().sort_index())


zcta_bus_weights: 1933 ZCTAs -> 6329 buses, 14493 pairs (mean 7.50 buses/ZCTA, max 43)
  ZCTAs split across >1 bus: 1744 / 1933 (90.2%)
  previous nearest-bus file reached 1,598 buses in total

ZCTAs per zone (majority weight):
iecc_zone
2A    884
2B     62
3A    595
3B    288
4B    104
Name: count, dtype: int64


#### Summary (updated)

- `county_to_iecc_zone.csv` — 254 counties -> 5 IECC zones (2A/2B/3A/3B/4B), no county split across zones.
- **`bus_zone_master.csv`** — all 6,717 Texas 7k buses: zone, county, lat/lon, kV, case `Load_MW`/`Gen_MW`,
  ERCOT `Area_Name`. Authoritative bus->zone; 100% agreement with the NREL county crosswalk.
- **`tract_bus_weights.csv`** — (GEOID, Bus_Number, weight) area-split table, 6,896 tracts ->
  6,359 buses over 24,499 pairs, weights summing to 1 per tract (50 equal-area parts each).
- **`bus_building_counts_statewide.csv`** — 5 ResStock archetype counts per bus (fractional),
  11,628,419 units = 100% of the Texas census total.
- **`zcta_bus_weights.csv`** — (ZCTA5CE10, Bus_Number, weight) area-split table, 1,933 ZCTAs ->
  6,329 buses (100 equal-area parts each). Replaces the nearest-bus ZCTA join, which reached 1,598.
- `zone_county_weather_weights.csv` — county weights per zone, from the area-split allocation.
- `original_coast_counties.csv` — the 12 counties for the Coast cross-check.
- `zcta_to_zone.csv` — one majority-weight zone label per ZCTA.

**Caveat to state in the paper**: both splits are *equal-area*, so they assume households (and
commercial floor space) are uniformly distributed inside a tract or ZCTA. That is a weak
assumption in large rural polygons with one town, and near-harmless in dense urban ones, which is
where most load sits. It is still strictly better than assigning 100% of a polygon to one bus,
and `Load_MW` in `bus_zone_master.csv` gives a per-bus reference to quantify the residual error.